####  0. Setup e carga do JSON

In [32]:
import pandas as pd
import json

path = '../data/weather_data.json'

with open(path) as f:
    raw = json.load(f)
    
print("Dados carregados com sucesso!")

Dados carregados com sucesso!


#### 1. Flattening (Achatando o JSON)

In [33]:
# json_normalize achata dicionários aninhados automaticamente.
# record_path='weather' extrai o primeiro (e único) item da lista.
# record_prefix adiciona 'weather.' nas colunas da lista para não dar conflito com o 'id' da cidade.

df = pd.json_normalize(
    raw,
    record_path='weather',          
    record_prefix='weather.',       # <--- A MÁGICA ACONTECE AQUI
    meta=[
        'base', 'visibility', 'dt', 'timezone', 'id', 'name', 'cod',
        ['coord', 'lon'], ['coord', 'lat'],
        ['main', 'temp'], ['main', 'feels_like'], ['main', 'temp_min'],
        ['main', 'temp_max'], ['main', 'pressure'], ['main', 'humidity'],
        ['wind', 'speed'], ['wind', 'deg'], ['wind', 'gust'],
        ['clouds', 'all'],
        ['sys', 'country'], ['sys', 'sunrise'], ['sys', 'sunset']
    ],
    errors='ignore'   
)

# rain vem separado — json_normalize não entra nela via record_path
# Precisamos achatar manualmente e adicionar no DataFrame final
rain_df = pd.json_normalize(raw).get('rain.1h')  

if rain_df is not None:
    df['rain.1h'] = rain_df.values[0]

df.head()

,weather.id,weather.main,weather.description,weather.icon,base,visibility,dt,timezone,id,name,...,main.pressure,main.humidity,wind.speed,wind.deg,wind.gust,clouds.all,sys.country,sys.sunrise,sys.sunset,rain.1h
0,500,Rain,chuva leve,10d,stations,6424,1776880187,-10800,3450554,Salvador,...,1012,78,6.82,161,7.54,57,BR,1776847219,1776889466,0.37


#### 2. Conversão de Timestamps e Fuso Horário

In [ ]:
# A API entrega tudo em Unix (segundos desde 1970) em UTC.
# unit='s' converte para datetime, depois localizamos e ajustamos o fuso.

for col in ['dt', 'sys.sunrise', 'sys.sunset']:
    df[col] = (
        pd.to_datetime(df[col], unit='s')
          .dt.tz_localize('UTC')
          .dt.tz_convert('America/Bahia')  # UTC-3, respeita horário de verão
    )

df[['dt', 'sys.sunrise', 'sys.sunset']]

,dt,sys.sunrise,sys.sunset
0,2026-04-22 14:49:47-03:00,2026-04-22 05:40:19-03:00,2026-04-22 17:24:26-03:00


#### 3. Validação de Schema

In [35]:
# rain.1h só existe no JSON quando está chovendo.
# Se não existir → cria a coluna zerada.
# Se existir mas com NaN → preenche com 0 (sem chuva = 0mm).

if 'rain.1h' not in df.columns:
    df['rain.1h'] = 0.0
else:
    df['rain.1h'] = df['rain.1h'].fillna(0.0)

print(df['rain.1h'].values)  # deve mostrar 0.37 neste caso

[0.37]


####  4. Limpeza e Padronização (Camada Silver)

In [36]:
# Colunas que não vão para o banco Silver
colunas_drop = ['base', 'id', 'cod', 'sys.id', 'sys.type', 'weather.icon', 'weather.id']
df = df.drop(columns=[c for c in colunas_drop if c in df.columns])

# Mapeamento para o padrão
rename_map = {
    'name':           'cidade',
    'sys.country':    'pais',
    'coord.lat':      'latitude',
    'coord.lon':      'longitude',
    'dt':             'data_hora',
    'main.temp':      'temperatura_c',
    'main.feels_like':'sensacao_termica_c',
    'main.temp_min':  'temp_min_c',
    'main.temp_max':  'temp_max_c',
    'main.pressure':  'pressao_hpa',
    'main.humidity':  'umidade_pct',
    'wind.speed':     'vento_velocidade_ms',
    'wind.deg':       'vento_direcao_grau',
    'wind.gust':      'vento_rajada_ms',
    'rain.1h':        'chuva_1h_mm',
    'clouds.all':     'nuvens_pct',
    'visibility':     'visibilidade_m',
    
    # Adicionado o prefixo 'weather.' nessas duas chaves
    'weather.main':        'condicao_clima',       
    'weather.description': 'descricao_clima',
    
    'sys.sunrise':    'nascer_sol',
    'sys.sunset':     'por_sol',
}

df = df.rename(columns=rename_map)
df.columns.tolist()

['condicao_clima',
 'descricao_clima',
 'visibilidade_m',
 'data_hora',
 'timezone',
 'cidade',
 'longitude',
 'latitude',
 'temperatura_c',
 'sensacao_termica_c',
 'temp_min_c',
 'temp_max_c',
 'pressao_hpa',
 'umidade_pct',
 'vento_velocidade_ms',
 'vento_direcao_grau',
 'vento_rajada_ms',
 'nuvens_pct',
 'pais',
 'nascer_sol',
 'por_sol',
 'chuva_1h_mm']

#### 5.  Enriquecimento e Vetorização

In [37]:
# np.select é mais eficiente que .apply() em DataFrames grandes.
# A ordem importa: a primeira condição verdadeira vence.

chuva  = df['chuva_1h_mm']
vento  = df['vento_velocidade_ms']
umidade = df['umidade_pct']

condicoes = [
    (chuva >= 50) | (vento >= 15),           # CRÍTICO
    (chuva >= 25),                           # ALERTA
    (chuva >= 5) & (umidade >= 80),          # ATENÇÃO
]
valores = ['CRÍTICO', 'ALERTA', 'ATENÇÃO']

df['nivel_risco'] = np.select(condicoes, valores, default='NORMAL')

df[['cidade', 'temperatura_c', 'chuva_1h_mm', 'vento_velocidade_ms',
    'umidade_pct', 'nivel_risco']]

,cidade,temperatura_c,chuva_1h_mm,vento_velocidade_ms,umidade_pct,nivel_risco
0,Salvador,26.48,0.37,6.82,78,NORMAL


#### 6.  Exportação e Check de Qualidade Final

In [38]:
# Salva o Silver temporariamente para validar antes de modularizar
output = Path('../data/weather_silver.json')
df.to_json(output, orient='records', indent=4, date_format='iso')

print(f"Shape: {df.shape}")
print(df.dtypes)

Shape: (1, 23)
condicao_clima                                  str
descricao_clima                                 str
visibilidade_m                               object
data_hora              datetime64[s, America/Bahia]
timezone                                     object
cidade                                          str
longitude                                    object
latitude                                     object
temperatura_c                                object
sensacao_termica_c                           object
temp_min_c                                   object
temp_max_c                                   object
pressao_hpa                                  object
umidade_pct                                  object
vento_velocidade_ms                          object
vento_direcao_grau                           object
vento_rajada_ms                              object
nuvens_pct                                   object
pais                                            s